# **Neural Machine Translation with Language Models**
In this lab, we build a complete Neural Machine Translation (NMT) pipeline using a multilingual sequence-to-sequence (Seq2Seq) language model. The goal is to fine-tune a pre-trained BART-based model on a bilingual Tay–Vietnamese dataset and evaluate its translation quality using BLEU and METEOR metrics.

# **Install libraries & import modules**
This cell installs all required packages (Transformers, Datasets, Torch, Evaluate).
These provide tools for tokenization, model loading, training, BLEU scoring, etc.

In [ ]:
!pip install sacrebleu transformers datasets torch evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00


# **Define model checkpoint, dataset name, and sequence lengths**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd
import numpy as np
import re
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)
import evaluate
import os

MODEL_CKPT = "FiveC/BartTay"

TRAIN_PATH = "/content/drive/MyDrive/Lab6-BARTpho/train.xlsx"
TEST_PATH  = "/content/drive/MyDrive/Lab6-BARTpho/test.xlsx"

train_df = pd.read_excel(TRAIN_PATH)
test_df  = pd.read_excel(TEST_PATH)

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

# **Data Cleaning Functions**
This cell defines clean_text() to remove:

+ special characters

+ newlines, tabs

+ extra whitespace

Clean data helps the model learn better and prevents noise from damaging the translation quality.

In [ ]:
train_df = train_df.dropna(subset=["tay", "viet"])
test_df  = test_df.dropna(subset=["tay", "viet"])

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return None
    text = re.sub(r'\\n|\|\\|[\n\r\t]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Áp dụng cleaning cho từng dòng
train_df["tay"]  = train_df["tay"].astype(str).apply(clean_text)
train_df["viet"] = train_df["viet"].astype(str).apply(clean_text)

test_df["tay"]  = test_df["tay"].astype(str).apply(clean_text)
test_df["viet"] = test_df["viet"].astype(str).apply(clean_text)

print(train_df.head())
print(test_df.head())

# **Load tokenizer and model**

In [ ]:
print(f"Loading tokenizer & model from {MODEL_CKPT} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CKPT)

# **Preprocessing function**
This function tokenizes Tay sentences as inputs and Vietnamese sentences as targets.

Key steps:

+ truncate & pad inputs

+ tokenize targets using as_target_tokenizer()

+ attach labels to model inputs

The output format matches what Seq2SeqTrainer expects.

In [ ]:
def preprocess_function(examples):
    inputs = examples["tay"]
    targets = examples["viet"]

    # Tokenize input
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding="max_length"
    )

    # Tokenize target
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)

In [ ]:
cols_to_remove = ["__index_level_0__"]
train_dataset = train_dataset.remove_columns([c for c in cols_to_remove if c in train_dataset.column_names])
test_dataset  = test_dataset.remove_columns([c for c in cols_to_remove if c in test_dataset.column_names])

In [ ]:
train_dataset = train_dataset.map(preprocess_function, batched=True)
test_dataset  = test_dataset.map(preprocess_function, batched=True)

# **Define SacreBLEU metric**

In [ ]:

import evaluate
sacrebleu = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    """
    - Loại bỏ token đặc biệt, strip(), ...
    - pred: list[str], label: list[str].
    """
    preds = [pred.strip() for pred in preds]
    labels = [lbl.strip() for lbl in labels]
    return preds, labels

def compute_metrics(eval_preds):
    """
    eval_preds = (predictions, labels)
    """
    preds, labels = eval_preds

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    pred_str = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    pred_str, label_str = postprocess_text(pred_str, label_str)

    references = [[l] for l in label_str]
    results = sacrebleu.compute(predictions=pred_str, references=references)
    return {"sacrebleu": results["score"]}

# **Configure training arguments**

In [ ]:
BATCH_SIZE = 16

training_args = Seq2SeqTrainingArguments(
    f"BartTayFinal",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="sacrebleu",
    greater_is_better=True,
    save_total_limit=3,
    fp16=True,
)

In [ ]:

from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
from accelerate import Accelerator
accelerator = Accelerator()

print("Starting Training ...")
trainer.train()

**Train the model using three different optimizers and compare their impact on translation performance:**

+ AdamW (default)

+ Adafactor (recommended for Seq2Seq models)

+ RMSprop

**For each optimizer:**

+ Keep all hyperparameters the same except for the optimizer

+ Train for the same number of epochs

+ Evaluate the model on the test set

+ Record and compare BLEU, chrF, and training stability

**Analysis should consider:**

+ Which optimizer achieves the best performance

+ Which optimizer converges faster

+ Which optimizer is more stable, especially for low-resource languages like Tay